In [1]:
import tensorflow as tf
from tensorflow.keras import layers, Model

2026-03-12 08:38:47.793850: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-03-12 08:38:48.822089: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI AVX512_BF16 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2026-03-12 08:38:54.810334: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.


In [2]:
def classification_head(x):
    # GAP: average tamper evidence across the whole image
    # GMP: strongest localised tamper signal (handles small splices/clones)
    gap = layers.GlobalAveragePooling2D(name='cls_gap')(x)
    gmp = layers.GlobalMaxPooling2D(name='cls_gmp')(x)
    x   = layers.Concatenate(name='cls_pool_cat')([gap, gmp])

    x = layers.Dense(1024, use_bias=False, name='cls_fc1')(x)
    x = layers.BatchNormalization(name='cls_bn1')(x)
    x = layers.ReLU(name='cls_relu1')(x)
    x = layers.Dropout(0.4, name='cls_drop1')(x)

    x = layers.Dense(512, use_bias=False, name='cls_fc2')(x)
    x = layers.BatchNormalization(name='cls_bn2')(x)
    x = layers.ReLU(name='cls_relu2')(x)
    x = layers.Dropout(0.3, name='cls_drop2')(x)

    x = layers.Dense(1, activation='sigmoid', name='cls_output')(x)
    return x


In [3]:
# Input: fused output from final_fusion() — noise + texture-frequency streams
inputs    = layers.Input(shape=(None, None, 896), name='cls_input')
outputs   = classification_head(inputs)
cls_model = Model(inputs=inputs, outputs=outputs, name='ClassificationHead')
cls_model.summary()


I0000 00:00:1773284055.394244    9613 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 4164 MB memory:  -> device: 0, name: NVIDIA GeForce RTX 3050 6GB Laptop GPU, pci bus id: 0000:01:00.0, compute capability: 8.6
2026-03-12 08:39:16.164629: W external/local_xla/xla/service/gpu/llvm_gpu_backend/default/nvptx_libdevice_path.cc:41] Can't find libdevice directory ${CUDA_DIR}/nvvm/libdevice. This may result in compilation or runtime failures, if the program we try to run uses routines from libdevice.
Searched for CUDA in the following directories:
  ./cuda_sdk_lib
  ipykernel_launcher.runfiles/cuda_nvcc
  ipykernel_launcher.runfiles/cuda_nvdisasm
  ipykernel_launcher.runfiles/nvidia_nvshmem
  ipykern/cuda_nvcc
  ipykern/cuda_nvdisasm
  ipykern/nvidia_nvshmem
  
  /usr/local/cuda
  /opt/cuda
  /home/kiran/Desktop/Tampering-Detection/venv/lib/python3.12/site-packages/tensorflow/python/platform/../../../nvidia/cuda_nvcc
  /home/kiran/Desktop/Tampering-Detec

Model: "ClassificationHead"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ cls_input           │ (None, None,      │          0 │ -                 │
│ (InputLayer)        │ None, 896)        │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ cls_gap             │ (None, 896)       │          0 │ cls_input[0][0]   │
│ (GlobalAveragePool… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ cls_gmp             │ (None, 896)       │          0 │ cls_input[0][0]   │
│ (GlobalMaxPooling2… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ cls_pool_cat        │ (None, 1792)      │          0 │ cls_gap[0][0],    │
│ (Concatenate)       │                   │            │ cls_gmp[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ cls_fc1 (Dense)     │ (None, 1024)      │  1,835,008 │ cls_pool_cat[0][… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ cls_bn1             │ (None, 1024)      │      4,096 │ cls_fc1[0][0]     │
│ (BatchNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ cls_relu1 (ReLU)    │ (None, 1024)      │          0 │ cls_bn1[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ cls_drop1 (Dropout) │ (None, 1024)      │          0 │ cls_relu1[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ cls_fc2 (Dense)     │ (None, 512)       │    524,288 │ cls_drop1[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ cls_bn2             │ (None, 512)       │      2,048 │ cls_fc2[0][0]     │
│ (BatchNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ cls_relu2 (ReLU)    │ (None, 512)       │          0 │ cls_bn2[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ cls_drop2 (Dropout) │ (None, 512)       │          0 │ cls_relu2[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ cls_output (Dense)  │ (None, 1)         │        513 │ cls_drop2[0][0]   │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 2,365,953 (9.03 MB)

 Trainable params: 2,362,881 (9.01 MB)

 Non-trainable params: 3,072 (12.00 KB)